In [1]:
import pandas as pd
df = pd.read_csv("/content/ev_charging_patterns.csv")
df.shape

(1320, 20)

In [3]:
df.isnull().sum()

,0
User ID,0
Vehicle Model,0
Battery Capacity (kWh),0
Charging Station ID,0
Charging Station Location,0
Charging Start Time,0
Charging End Time,0
Energy Consumed (kWh),66
Charging Duration (hours),0
Charging Rate (kW),66


In [6]:
df.nunique()

,0
User ID,1320
Vehicle Model,5
Battery Capacity (kWh),147
Charging Station ID,462
Charging Station Location,5
Charging Start Time,1320
Charging End Time,1309
Energy Consumed (kWh),1254
Charging Duration (hours),1320
Charging Rate (kW),1254


In [5]:
df.dtypes

,0
User ID,object
Vehicle Model,object
Battery Capacity (kWh),float64
Charging Station ID,object
Charging Station Location,object
Charging Start Time,object
Charging End Time,object
Energy Consumed (kWh),float64
Charging Duration (hours),float64
Charging Rate (kW),float64


In [7]:
df.drop(columns=['User ID','Charging Station ID'],inplace=True)

In [8]:
df.shape

(1320, 18)

In [9]:
df.drop_duplicates(inplace=True)

In [10]:
df.shape

(1320, 18)

In [11]:
df.dropna(inplace=True)

In [12]:
df.shape

(1131, 18)

In [14]:
df.drop(columns=['Charging Start Time',
                 'Charging End Time'],inplace=True)

In [15]:
df.shape

(1131, 16)

In [16]:
df_ohe = pd.get_dummies(df)
df_ohe.shape

(1131, 37)

In [18]:
X = df_ohe

In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test = train_test_split(X,
                                  test_size=0.3,
                                  random_state=7)
X_train.shape,X_test.shape

((791, 37), (340, 37))

In [20]:
X_train = (X_train - X_train.mean()) / X_train.std()

In [21]:
X_test = (X_test- X_train.mean()) / X_train.std()

In [22]:
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(kernel="poly",degree=2,random_state=7)
kpca.fit(X_train)

KernelPCA(degree=2, kernel='poly', random_state=7)

In [23]:
kpca.eigenvalues_.shape

(474,)

In [24]:
from sklearn.decomposition import KernelPCA
kpca = KernelPCA(kernel="rbf",random_state=7)
kpca.fit(X_train)

KernelPCA(kernel='rbf', random_state=7)

In [25]:
kpca.eigenvalues_.shape

(790,)

In [26]:
def outlier_imputation_iqr(df,col):
  df1 = df.copy()
  q1,q3 = df1[col].quantile([0.25,0.75])
  iqr = q3-q1
  LB = q1 - 1.5*iqr
  UB = q3 + 1.5*iqr
  print("Column -->",col)
  df1.loc[(df1[col]<LB),col] = LB
  df1.loc[(df1[col]>UB),col] = UB
  print(LB,UB, df1[col].min(),df1[col].max())
  return df1

In [27]:
con_col = X_train.columns[X_train.nunique()>15]
con_col

Index(['Battery Capacity (kWh)', 'Energy Consumed (kWh)',
       'Charging Duration (hours)', 'Charging Rate (kW)',
       'Charging Cost (USD)', 'State of Charge (Start %)',
       'State of Charge (End %)', 'Distance Driven (since last charge) (km)',
       'Temperature (°C)', 'Vehicle Age (years)'],
      dtype='object')

In [28]:
X_train_io = X_train.copy()
for col in con_col:
  X_train_io = outlier_imputation_iqr(X_train_io,col)

Column --> Battery Capacity (kWh)
-2.3241696227235886 2.228495421995734 -2.3241696227235886 2.228495421995734
Column --> Energy Consumed (kWh)
-3.3271059378496632 3.3446660603796365 -1.950026882211301 3.3446660603796365
Column --> Charging Duration (hours)
-3.2682970989850366 3.2263772904346064 -2.0676213924362337 3.2263772904346064
Column --> Charging Rate (kW)
-3.309043836822947 3.264113001800868 -1.7479735039452566 3.264113001800868
Column --> Charging Cost (USD)
-3.4218791643536455 3.4333135841573217 -1.9756361885336213 3.4333135841573217
Column --> State of Charge (Start %)
-3.4419401990972824 3.39951514063654 -1.9107112674418218 3.041918129848564
Column --> State of Charge (End %)
-3.1149311902721966 3.1267312251564077 -3.1149311902721966 3.1267312251564077
Column --> Distance Driven (since last charge) (km)
-3.4765358075685264 3.5057930797496843 -1.7746298502964073 2.8085617566921934
Column --> Temperature (°C)
-3.304466717677972 3.325483864301752 -1.744866625970943 3.3254838643